In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import sionna.rt as rt
from utils.scene_utils import add_txs
from utils.geo_coords import SceneCoordinateConverter
from config import FILE_ENCODE, LocalCRS

# Define the geographical boundary of the target region:
# I take the max and min latitude and longitude of transmitters and extend it with 500m
# Extended Formula:
# latitude 1° is approximately equal to 111,000m
# 500 / 111 000 ≈ 0.004505
# 48.90138888888889 + 0.004505 ≈ 48.9059; 48.818333333333335 - 0.004505 ≈ 48.8138
# longitude 1° is approximately equal to $\Delta_{lon} =\Delta_{lat} \times \cos(lat)$
# 111 000 * cos((48.9014 + 48.8183)/2) ≈ 111 000 * cos(48.86) ≈ 111 000 * 0.6579 ≈ 73 000
# 500 / 73 000 ≈ 0.006849
# 2.249722222222222 - 0.006849 ≈ 2.2429 ; 2.450555555555556 + 0.006849 ≈ 2.4574
LAT_MAX, LAT_MIN = 48.9059, 48.8138
LON_MIN, LON_MAX = 2.2429, 2.4574

# Calculate the center origin point of the scene
LAT_ORIGIN=(LAT_MAX + LAT_MIN)/2
LON_ORIGIN=(LON_MIN + LON_MAX)/2
TRANSMITTER_DIRECTORY = Path('data/transmitters')

In [ ]:
# Optional: Preprocess antenna files. Filter antennas by frequence and postal code
from utils.preprocess_raw_data import preprocess_antenna_data_by_frequency_and_postal

ANTENNES_INFO_FILENAME = Path('data/paris/Antennes_Emetteurs_Bandes_Cartoradio.csv')
ANTENNES_LOC_FILENAME = Path('data/paris/Sites_Cartoradio.csv')
FILTER_FREQUENCE = 2600
FILTER_POSTAL_CODE = r'^75\d{3}$'

preprocess_antenna_data_by_frequency_and_postal(
    ANTENNES_INFO_FILENAME, 
    ANTENNES_LOC_FILENAME, 
    FILTER_FREQUENCE, 
    FILTER_POSTAL_CODE, 
    TRANSMITTER_DIRECTORY)

# Step 1: Region Grid Splitting
In this step, we split the large Paris area into smaller blocks (e.g., 2.56km x 2.56km). 
This helps our computer run Sionna simulation without running out of memory.

**Key features:**
* Use official French national projection - **Lambert-93** to work with meters.
* Add an **overlap buffer** (e.g., 150m) to avoid edge errors in ray-tracing.

In [ ]:
# Import the splitter class from map_splitter.py
from utils.map_splitter import TileSplitter

# Initialize the splitter
# Each block is 2560m x 2560m. Overlap is 150m.
splitter = TileSplitter(
    lat_min=LAT_MIN, 
    lat_max=LAT_MAX, 
    lon_min=LON_MIN, 
    lon_max=LON_MAX, 
    block_size_m=2560, 
    overlap_m=150
)

# Get the task list for loops later
all_blocks = splitter.get_all_blocks()

In [ ]:
# DEBUG
all_blocks = all_blocks[0:1]

# Step 2: Per-Block PLY & XML Generation

In this step, we iterate over each grid block and generate the 3D scene files required by Sionna ray-tracing. For each block we produce **all available layer** PLY mesh files (buildings, roads, railways, water, forest) plus terrain, and one Mitsuba XML scene file.

**Key features:**
* Fetch OSM data per block using `OSMFetcher` with the block's **overlap-extended** bounding box, ensuring consistent coverage with the terrain.
* Convert OSM layer footprints (buildings, roads, railways, water, forest) to 3D meshes via `OSMToPLY`. Buildings **without height information are dropped** to avoid unrealistic flat geometry; other layers use a default height of `0.0`.
* Generate a flat terrain mesh at `z = 0.0` with **10 m resolution** covering the full overlap-extended Lambert-93 extent.
* Place all PLY files under a `meshes/` subdirectory alongside the XML, matching Sionna's expected path convention (`meshes/<filename>.ply`).
* Assemble the final XML scene with `SionnaXMLGenerator`, referencing all available mesh layers. Layers with no features still produce a minimal valid PLY file, keeping the XML structure consistent across blocks.

In [ ]:
from pathlib import Path

XML_DIR = Path('data/xml')  # Root directory for all block outputs
TERRAIN_RESOLUTION = 10.0   # Grid cell size in meters
TERRAIN_HEIGHT = 0.0        # Constant Z height for terrain vertices

# =============================================================================
# Layer Definitions
# Each layer maps an OSM tag set to its output PLY filename and default height.
# Buildings use 'drop' mode: polygons without height info are skipped.
# Other layers use default_height=0.0 (flat geometry at ground level).
# =============================================================================
LAYERS = [
    {
        "tag_name": "buildings",
        "ply_filename": "buildings.ply",
        "default_height": 0.0,       # Not used for buildings (see handle_missing_height)
        "handle_missing_height": "drop",
    },
    {
        "tag_name": "roads",
        "ply_filename": "roads.ply",
        "default_height": 0.1,
        "handle_missing_height": "use_default",
    },
    {
        "tag_name": "railways",
        "ply_filename": "railways.ply",
        "default_height": 0.5,
        "handle_missing_height": "use_default",
    },
    {
        "tag_name": "water",
        "ply_filename": "water.ply",
        "default_height": 0.2,
        "handle_missing_height": "use_default",
    },
    {
        "tag_name": "forest",
        "ply_filename": "forest.ply",
        "default_height": 2.0,
        "handle_missing_height": "use_default",
    },
]

In [ ]:
from utils.osm_fetcher import OSMFetcher, OSM_TAGS
from utils.osm_to_ply import OSMToPLY
from utils.osm_to_ply import generate_flat_terrain_ply
from utils.generate_xml import SionnaXMLGenerator

for block_info in all_blocks:
    row = block_info["row"]
    col = block_info["col"]
    block_name = block_info["name"]
    
    print(f"\n{'='*60}")
    print(f"Processing {block_name} (row={row}, col={col})")
    print(f"{'='*60}")
    
    # --- Get block metadata (with overlap for consistent coverage) ---
    meta = splitter.get_block_latlon_bounds(row, col)
    
    # --- Create output directories ---
    block_dir = XML_DIR / block_name
    mesh_dir = block_dir / "meshes"
    mesh_dir.mkdir(parents=True, exist_ok=True)
    
    # --- Fetch OSM data for this block ---
    # IMPORTANT: bbox order is (left, bottom, right, top)
    # which corresponds to (lon_min, lat_min, lon_max, lat_max)
    fetcher = OSMFetcher(
        bbox=(meta.lon_min, meta.lat_min, meta.lon_max, meta.lat_max),
        tags=OSM_TAGS["complete"],
    )
    
    # --- Generate PLY for each layer ---
    for layer in LAYERS:
        tag_name = layer["tag_name"]
        ply_filename = layer["ply_filename"]
        ply_path = mesh_dir / ply_filename
        
        # Filter the pre-fetched data for this specific layer
        gdf = fetcher.get_filtered_features(OSM_TAGS[tag_name])
        
        if gdf.empty:
            print(f"  [{tag_name}] No features found — writing empty PLY")
        
        # Convert OSM geometries to 3D PLY mesh
        converter = OSMToPLY(
            gdf=gdf,
            ply_path=ply_path,
            default_height=layer["default_height"],
        )
        converter._process_polygons(handle_missing_height=layer["handle_missing_height"])
        converter._collect_3d_polygons()
        converter._build_multi_polygon()
        converter.save_to_ply()
        
        print(f"  [{tag_name}] Saved to {ply_path}")
    
    # --- Generate terrain PLY ---
    # Uses the overlap-extended Lambert-93 coordinates to match OSM coverage
    terrain_path = mesh_dir / "terrain.ply"
    generate_flat_terrain_ply(
        output_path=terrain_path,
        x_min=meta.x_start,
        x_max=meta.x_end,
        y_min=meta.y_start,
        y_max=meta.y_end,
        resolution=TERRAIN_RESOLUTION,
        height=TERRAIN_HEIGHT,
    )
    print(f"  [terrain] Saved to {terrain_path}")
    
    # --- Generate Sionna XML scene file ---
    xml_path = block_dir / f"{block_name}.xml"
    xml_generator = SionnaXMLGenerator(mesh_dir=mesh_dir, output_path=xml_path)
    xml_generator.generate(validate_meshes=False)  # Allow missing layers (empty PLYs exist)
    print(f"  [xml] Saved to {xml_path}")
    
    print(f"Finished {block_name}")

print(f"\n{'='*60}")
print(f"All {len(all_blocks)} blocks processed successfully.")
print(f"Output directory: {XML_DIR.resolve()}")

# Step 3: Per-Block Radio Map Generation

In this step, we load each block's Mitsuba XML scene into Sionna 2.0.1 and compute a summed RSS radio map using ray-tracing. For each block, transmitters inside the **core area** (overlap excluded) are filtered from a global CSV and placed into the scene. The resulting radio map captures the combined received signal strength from all active transmitters over a dense 1 m grid.

**Key features:**
* Initialize a `SceneCoordinateConverter` per block with its origin at the **center of the overlap-extended bounding box** and altitude at `TERRAIN_HEIGHT`, matching the terrain surface.
* Load the block's XML scene via `rt.load_scene()` and configure it with a `FREQUENCY` carrier frequency and a single-element cross-polarized antenna array (`tr38901` pattern).
* Filter transmitters from `TRANSMITTER_PATH` to keep only those within the block's **core area** (no overlap), avoiding duplicate coverage across neighboring blocks.
* Update radio materials for water and roads to physically accurate ITU models (`freshwater`, `asphalt_concrete`) at the target frequency.
* Compute the radio map with `rt.RadioMapSolver` at `RM_RESOLUTION_M` over the full `block_size_m × block_size_m` core area, using up to 12 ray bounces (`max_depth=5`) and 10 million rays per transmitter.
* Sum the linear RSS contributions from all transmitters into a single 2D array of shape `(block_size_m, block_size_m)`. Blocks with **no transmitters** in their core area return `None` and are skipped with a log message.

In [ ]:
TRANSMITTER_PATH = Path('data/transmitters/2600_mhz.csv')  # Global transmitter file
FREQUENCY = 2.6e9          # Carrier frequency in Hz (2.6 GHz)
RM_RESOLUTION_M = 1.0  # Radio map cell size in meters

# =============================================================================
# Antenna Array
# Single-element cross-polarized antenna following TR 38.901 pattern.
# =============================================================================
tx_array = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="cross",
)

In [ ]:
from config import LocalCRS
from utils.geo_coords import SceneCoordinateConverter
from utils.generate_radiomap import RadioMapGenerator

for block_info in all_blocks:
    row = block_info["row"]
    col = block_info["col"]
    block_name = block_info["name"]
    
    print(f"\n{'='*60}")
    print(f"Processing {block_name} (row={row}, col={col})")
    print(f"{'='*60}")
    
    # --- Get block metadata ---
    meta = splitter.get_block_latlon_bounds(row, col)
    
    # --- Paths for this block ---
    block_dir = XML_DIR / block_name
    xml_path = block_dir / f"{block_name}.xml"
    
    if not xml_path.exists():
        print(f"  [skip] XML file not found: {xml_path}")
        continue
    
    # --- Initialize coordinate converter for this block ---
    # Origin is the center of the overlap-extended bounding box, at z = TERRAIN_HEIGHT
    # (matching the terrain surface height used in Step 2).
    lat_origin = (meta.lat_min + meta.lat_max) / 2.0
    lon_origin = (meta.lon_min + meta.lon_max) / 2.0
    converter = SceneCoordinateConverter(
        lat_origin,
        lon_origin,
        TERRAIN_HEIGHT,                    # ALT_ORIGIN matches terrain height
        LocalCRS.OSM_STORAGE.crs,
        LocalCRS.FRANCE_LAMBERT93.crs,
    )
    
    # --- Generate radio map ---
    generator = RadioMapGenerator(converter)
    
    rss_map = generator.generate(
        xml_path=xml_path,
        csv_path=TRANSMITTER_PATH,
        block_meta=meta,
        tx_array=tx_array,
        frequency=FREQUENCY,
        resolution_m=RM_RESOLUTION_M,
    )
    
    # --- Check result ---
    if rss_map is None:
        print(f"  [skip] No transmitters in core area of {block_name}")
    else:
        print(f"  [done] RSS map shape: {rss_map.shape}, "
              f"dtype: {rss_map.dtype}, "
              f"min: {rss_map.min():.6e}, max: {rss_map.max():.6e}")

print(f"\n{'='*60}")
print(f"All {len(all_blocks)} blocks processed.")